## 1. Импорт библиотек

In [1]:
from pathlib import Path
from io import BytesIO
import json
import math
import random
import warnings

import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, Subset

try:
    import dagshub
    import mlflow
except ImportError as error:
    raise ImportError(
        "Не найдены dagshub или mlflow. Установите зависимости: pip install dagshub mlflow"
    ) from error

try:
    from transformers import (
        AutoProcessor,
        Trainer,
        TrainerCallback,
        TrainingArguments,
        Qwen2_5_VLForConditionalGeneration,
        set_seed,
    )
except ImportError as error:
    raise ImportError(
        "Не найден transformers с поддержкой Qwen2.5-VL. "
        "Установите или обновите пакет: pip install -U transformers accelerate"
    ) from error

try:
    from peft import LoraConfig, TaskType, get_peft_model
except ImportError as error:
    raise ImportError(
        "Не найден peft. Установите пакет: pip install peft"
    ) from error

try:
    from qwen_vl_utils import process_vision_info
except ImportError as error:
    raise ImportError(
        "Не найден qwen-vl-utils. Установите пакет: pip install qwen-vl-utils"
    ) from error


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Конфигурация проекта

Настройки ниже рассчитаны на запуск из корня проекта или из папки `notebooks`. Данные не скачиваются автоматически: сначала используется локальный parquet-формат GQA-ru из существующих ноутбуков, затем jsonl-файл `data/processed/train.jsonl` как запасной вариант.

In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень проекта относительно текущей директории")


PROJECT_ROOT = find_project_root()

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
LOCAL_MODEL_DIR = PROJECT_ROOT / "models" / "Qwen2.5-VL-3B-Instruct"
MODEL_SOURCE = LOCAL_MODEL_DIR if LOCAL_MODEL_DIR.exists() else MODEL_NAME
LOCAL_FILES_ONLY = LOCAL_MODEL_DIR.exists()

DATA_DIR = PROJECT_ROOT / "data" / "GQA-ru"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train.jsonl"
GQA_TRAIN_INSTRUCTIONS_DIR = DATA_DIR / "train_balanced_instructions"
GQA_TRAIN_IMAGES_DIR = DATA_DIR / "train_balanced_images"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "qwen_lora_adapter_3"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "qwen_lora_training_v3"
LOG_DIR = PROJECT_ROOT / "logs" / "qwen_lora_training_v3"
for directory in [OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_TRAIN_SAMPLES = 30000
MAX_LENGTH = 3072
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 640 * 28 * 28
GRADIENT_CHECKPOINTING = True

PROMPT_TEMPLATE = """Ответь на вопрос, используя изображение.
Дай только краткий ответ без объяснений.
Вопрос: {question}
Ответ:"""

DAGSHUB_REPO_OWNER = "Dezurn"
DAGSHUB_REPO_NAME = "vk-vision-language-vqa"
MLFLOW_EXPERIMENT_NAME = "qwen_lora_training_3"
MLFLOW_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_REPO_OWNER}/{DAGSHUB_REPO_NAME}.mlflow"

set_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

if not DATA_DIR.exists() and not DATA_PATH.exists():
    raise FileNotFoundError(
        "Не найдены локальные данные. Ожидается data/GQA-ru или data/processed/train.jsonl"
    )

if not LOCAL_MODEL_DIR.exists():
    warnings.warn(
        f"Локальная папка модели не найдена: {LOCAL_MODEL_DIR}. "
        "Transformers попробует загрузить MODEL_NAME из Hugging Face Hub."
    )

config_df = pd.DataFrame([
    {"Параметр": "project_root", "Значение": str(PROJECT_ROOT)},
    {"Параметр": "model_name", "Значение": MODEL_NAME},
    {"Параметр": "model_source", "Значение": str(MODEL_SOURCE)},
    {"Параметр": "local_files_only", "Значение": LOCAL_FILES_ONLY},
    {"Параметр": "data_dir", "Значение": str(DATA_DIR)},
    {"Параметр": "jsonl_fallback", "Значение": str(DATA_PATH)},
    {"Параметр": "output_dir", "Значение": str(OUTPUT_DIR)},
    {"Параметр": "cuda_available", "Значение": torch.cuda.is_available()},
    {"Параметр": "gpu", "Значение": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
])
display(config_df)


,Параметр,Значение
0,project_root,c:\Users\user\Desktop\Практика\vk-vision-langu...
1,model_name,Qwen/Qwen2.5-VL-3B-Instruct
2,model_source,c:\Users\user\Desktop\Практика\vk-vision-langu...
3,local_files_only,True
4,data_dir,c:\Users\user\Desktop\Практика\vk-vision-langu...
5,jsonl_fallback,c:\Users\user\Desktop\Практика\vk-vision-langu...
6,output_dir,c:\Users\user\Desktop\Практика\vk-vision-langu...
7,cuda_available,True
8,gpu,NVIDIA GeForce RTX 3080


## 3. Подключение DagsHub и MLflow

In [3]:
dagshub.init(
    repo_owner=DAGSHUB_REPO_OWNER,
    repo_name=DAGSHUB_REPO_NAME,
    mlflow=True,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
MLFLOW_READY = True


Accessing as Dezurn

Initialized MLflow to track repo "Dezurn/vk-vision-language-vqa"

Repository Dezurn/vk-vision-language-vqa initialized!

## 4. Загрузка подготовленных данных

In [4]:
def read_parquet_folder(folder):
    folder = Path(folder)
    files = sorted(folder.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"В папке не найдены parquet-файлы: {folder}")

    parts = [pd.read_parquet(file) for file in files]
    return pd.concat(parts, ignore_index=True)


def find_column(df, candidates, required=True):
    lower_to_original = {str(col).lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]

    if required:
        raise ValueError(f"Не найдена ни одна из колонок {candidates}. Доступные колонки: {list(df.columns)}")
    return None


def resolve_image_path(image_path, search_roots):
    image_path = Path(image_path)
    if image_path.is_absolute():
        return image_path

    for root in search_roots:
        candidate = Path(root) / image_path
        if candidate.exists():
            return candidate
    return image_path


def to_pil_image(image_value, search_roots):
    if isinstance(image_value, Image.Image):
        return image_value.convert("RGB")

    if isinstance(image_value, dict):
        if image_value.get("bytes") is not None:
            return Image.open(BytesIO(image_value["bytes"])).convert("RGB")

        if image_value.get("path"):
            image_path = resolve_image_path(image_value["path"], search_roots)
            if not image_path.exists():
                raise FileNotFoundError(f"Изображение не найдено: {image_path}")
            return Image.open(image_path).convert("RGB")

    if isinstance(image_value, (bytes, bytearray)):
        return Image.open(BytesIO(image_value)).convert("RGB")

    if isinstance(image_value, (str, Path)):
        image_path = resolve_image_path(image_value, search_roots)
        if not image_path.exists():
            raise FileNotFoundError(f"Изображение не найдено: {image_path}")
        return Image.open(image_path).convert("RGB")

    raise TypeError(f"Unsupported image value type: {type(image_value)}")


class JsonlVQADataset(Dataset):
    """Датасет для запасного jsonl-формата: image_path, question, answer."""

    required_fields = {"image_path", "question", "answer"}

    def __init__(self, path, project_root, data_dir):
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"JSONL-файл не найден: {self.path}")

        self.search_roots = [self.path.parent, project_root, data_dir]
        self.records = []
        with self.path.open("r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                missing = self.required_fields - set(record)
                if missing:
                    raise ValueError(f"Строка {line_number}: отсутствуют поля {sorted(missing)}")
                record["_line_number"] = line_number
                self.records.append(record)

        if not self.records:
            raise ValueError(f"JSONL-файл пуст: {self.path}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        question = str(record["question"]).strip()
        answer = str(record["answer"]).strip()
        image = to_pil_image(record["image_path"], self.search_roots)

        return {
            "id": str(record.get("id", record["_line_number"])),
            "image_id": str(record.get("image_id", record.get("image_path", ""))),
            "image": image,
            "question": question,
            "answer": answer,
            "prompt": PROMPT_TEMPLATE.format(question=question),
        }


class GQARuParquetDataset(Dataset):
    """Датасет для parquet-структуры GQA-ru, уже используемой в EDA и baseline."""

    def __init__(self, instructions, images, data_dir, project_root):
        self.data_dir = Path(data_dir)
        self.project_root = Path(project_root)
        self.search_roots = [self.data_dir, self.project_root]

        question_col = find_column(instructions, ["question", "query", "prompt", "text"])
        answer_col = find_column(instructions, ["answer", "ground_truth", "gt_answer", "label"])
        instruction_id_col = find_column(instructions, ["id", "question_id", "questionId"], required=False)
        image_id_col = find_column(instructions, ["imageId", "image_id", "imageid", "img_id"], required=False)

        records = instructions.copy()
        if not images.empty:
            if image_id_col is None:
                raise ValueError("В инструкциях не найдена колонка image_id для связи с таблицей изображений")
            image_table_id_col = find_column(images, ["id", "imageId", "image_id", "imageid"])
            image_col = find_column(images, ["image", "img", "picture"])
            image_lookup = images.set_index(image_table_id_col)[image_col].to_dict()
            records["_image"] = records[image_id_col].map(image_lookup)
        else:
            image_col = find_column(records, ["image", "img", "picture"])
            records["_image"] = records[image_col]

        records["_question"] = records[question_col].astype(str).str.strip()
        records["_answer"] = records[answer_col].astype(str).str.strip()
        records["_id"] = records[instruction_id_col].astype(str) if instruction_id_col else [str(i) for i in range(len(records))]
        records["_image_id"] = records[image_id_col].astype(str) if image_id_col else ""
        records = records.dropna(subset=["_image", "_question", "_answer"]).reset_index(drop=True)

        if records.empty:
            raise ValueError("После подготовки parquet-данных не осталось обучающих примеров")

        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        question = str(row["_question"]).strip()
        answer = str(row["_answer"]).strip()
        image = to_pil_image(row["_image"], self.search_roots)

        return {
            "id": str(row["_id"]),
            "image_id": str(row["_image_id"]),
            "image": image,
            "question": question,
            "answer": answer,
            "prompt": PROMPT_TEMPLATE.format(question=question),
        }


def load_training_dataset():
    parquet_ready = GQA_TRAIN_INSTRUCTIONS_DIR.exists() and GQA_TRAIN_IMAGES_DIR.exists()
    if parquet_ready:
        instructions_df = read_parquet_folder(GQA_TRAIN_INSTRUCTIONS_DIR)
        images_df = read_parquet_folder(GQA_TRAIN_IMAGES_DIR)
        dataset = GQARuParquetDataset(instructions_df, images_df, DATA_DIR, PROJECT_ROOT)
        source_info = {
            "data_source": "gqa_ru_parquet",
            "instructions_dir": str(GQA_TRAIN_INSTRUCTIONS_DIR),
            "images_dir": str(GQA_TRAIN_IMAGES_DIR),
        }
        return dataset, source_info

    dataset = JsonlVQADataset(DATA_PATH, PROJECT_ROOT, DATA_DIR)
    source_info = {
        "data_source": "jsonl",
        "data_path": str(DATA_PATH),
    }
    return dataset, source_info


raw_train_dataset, DATA_SOURCE_INFO = load_training_dataset()

if MAX_TRAIN_SAMPLES is not None:
    sample_size = min(MAX_TRAIN_SAMPLES, len(raw_train_dataset))
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(
        len(raw_train_dataset),
        generator=generator,
    ).tolist()

    if MAX_TRAIN_SAMPLES is not None:
        indices = indices[:MAX_TRAIN_SAMPLES]

    train_dataset = Subset(raw_train_dataset, indices)
else:
    train_dataset = raw_train_dataset

load_df = pd.DataFrame([
    {"Параметр": "data_source", "Значение": DATA_SOURCE_INFO["data_source"]},
    {"Параметр": "raw_train_examples", "Значение": len(raw_train_dataset)},
    {"Параметр": "used_train_examples", "Значение": len(train_dataset)},
])
display(load_df)


,Параметр,Значение
0,data_source,gqa_ru_parquet
1,raw_train_examples,40000
2,used_train_examples,30000


## 5. Подготовка датасета для обучения

In [5]:
def build_training_messages(example, include_answer=True):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example["image"]},
                {"type": "text", "text": example["prompt"]},
            ],
        }
    ]

    if include_answer:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": example["answer"]}]})

    return messages


class QwenVQACollator:
    """Формирует батч Qwen2.5-VL и маскирует loss на пользовательском промпте."""

    def __init__(self, processor, max_length=None):
        self.processor = processor
        self.max_length = max_length

        if getattr(self.processor, "tokenizer", None) is not None:
            self.processor.tokenizer.padding_side = "right"
            if self.processor.tokenizer.pad_token is None:
                self.processor.tokenizer.pad_token = self.processor.tokenizer.eos_token

    def _processor_kwargs(self):
        kwargs = {
            "padding": True,
            "return_tensors": "pt",
        }
        if self.max_length is not None:
            kwargs.update({"truncation": True, "max_length": self.max_length})
        return kwargs

    def _encode_messages(self, messages_batch, add_generation_prompt):
        texts = [
            self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
            for messages in messages_batch
        ]
        image_inputs, video_inputs = process_vision_info(messages_batch)
        return self.processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            **self._processor_kwargs(),
        )

    def __call__(self, examples):
        full_messages = [build_training_messages(example, include_answer=True) for example in examples]
        prompt_messages = [build_training_messages(example, include_answer=False) for example in examples]

        inputs = self._encode_messages(full_messages, add_generation_prompt=False)
        prompt_inputs = self._encode_messages(prompt_messages, add_generation_prompt=True)

        labels = inputs["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        prompt_lengths = prompt_inputs["attention_mask"].sum(dim=1).tolist()
        for row_index, prompt_length in enumerate(prompt_lengths):
            labels[row_index, : int(prompt_length)] = -100

        inputs["labels"] = labels
        return inputs


## 6. Загрузка базовой модели и процессора

In [6]:
def hf_model_source():
    return str(MODEL_SOURCE) if isinstance(MODEL_SOURCE, Path) else MODEL_SOURCE


def load_processor():
    processor_kwargs = {
        "local_files_only": LOCAL_FILES_ONLY,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
    }
    try:
        return AutoProcessor.from_pretrained(hf_model_source(), **processor_kwargs)
    except TypeError:
        processor_kwargs.pop("min_pixels", None)
        processor_kwargs.pop("max_pixels", None)
        return AutoProcessor.from_pretrained(hf_model_source(), **processor_kwargs)


def load_qwen_model():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    elif torch.cuda.is_available():
        model_dtype = torch.float16
    else:
        model_dtype = torch.float32

    model_kwargs = {
        "local_files_only": LOCAL_FILES_ONLY,
    }
    if torch.cuda.is_available():
        model_kwargs["device_map"] =  "cuda" if torch.cuda.is_available() else "cpu"

    MODEL_CLASS = Qwen2_5_VLForConditionalGeneration

    try:
        return MODEL_CLASS.from_pretrained(
            hf_model_source(),
            dtype=model_dtype,
            **model_kwargs,
        )
    except TypeError:
        return MODEL_CLASS.from_pretrained(
            hf_model_source(),
            torch_dtype=model_dtype,
            **model_kwargs,
        )


processor = load_processor()
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"

model = load_qwen_model()
model.config.use_cache = False

model_info_df = pd.DataFrame([
    {"Параметр": "model_class", "Значение": model.__class__.__name__},
    {"Параметр": "processor_class", "Значение": processor.__class__.__name__},
    {"Параметр": "dtype", "Значение": str(next(model.parameters()).dtype)},
])
display(model_info_df)


Loading weights: 100%|██████████| 824/824 [00:02<00:00, 390.62it/s]


,Параметр,Значение
0,model_class,Qwen2_5_VLForConditionalGeneration
1,processor_class,Qwen2_5_VLProcessor
2,dtype,torch.bfloat16


## 7. Настройка LoRA

In [7]:
LORA_CONFIG = {
    "r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "qkv",
        "proj",
    ],
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM,
}

peft_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, peft_config)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

vision_lora_modules = [
    name
    for name, module in model.named_modules()
    if "visual" in name.lower() and hasattr(module, "lora_A")
]

if not vision_lora_modules:
    raise RuntimeError(
        "LoRA не подключилась к визуальному энкодеру. "
        "Проверьте названия модулей установленной версии transformers."
    )

print(f"LoRA подключена к {len(vision_lora_modules)} визуальным модулям.")
print("Первые визуальные LoRA-модули:")
for name in vision_lora_modules[:10]:
    print(" -", name)

model.print_trainable_parameters()

LoRA подключена к 161 визуальным модулям.
Первые визуальные LoRA-модули:
 - base_model.model.model.visual.patch_embed.proj
 - base_model.model.model.visual.blocks.0.attn.qkv
 - base_model.model.model.visual.blocks.0.attn.proj
 - base_model.model.model.visual.blocks.0.mlp.gate_proj
 - base_model.model.model.visual.blocks.0.mlp.up_proj
 - base_model.model.model.visual.blocks.0.mlp.down_proj
 - base_model.model.model.visual.blocks.1.attn.qkv
 - base_model.model.model.visual.blocks.1.attn.proj
 - base_model.model.model.visual.blocks.1.mlp.gate_proj
 - base_model.model.model.visual.blocks.1.mlp.up_proj
trainable params: 82,248,448 || all params: 3,836,871,424 || trainable%: 2.1436


## 8. Подготовка обучения

In [8]:
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
SAVE_STEPS = 250 
SAVE_TOTAL_LIMIT = 2
MAX_GRAD_NORM = 1.0
LR_SCHEDULER_TYPE = "cosine"
OPTIMIZER = "adamw_torch"

BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16 = torch.cuda.is_available() and not BF16
RUN_NAME = f"qwen2_5_vl_lora_seed_{SEED}"

training_args_base = {
    "output_dir": str(CHECKPOINT_DIR),
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "logging_steps": LOGGING_STEPS,
    "logging_strategy": "steps",
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "max_grad_norm": MAX_GRAD_NORM,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "optim": OPTIMIZER,
    "bf16": BF16,
    "fp16": FP16,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "remove_unused_columns": False,
    "dataloader_num_workers": 0,
    "report_to": [],
    "seed": SEED,
    "data_seed": SEED,
}

training_args_base.update({
    "eval_steps": 250,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
})

try:
    training_args = TrainingArguments(
        **training_args_base,
        eval_strategy="steps",
    )
except TypeError:
    training_args = TrainingArguments(
        **training_args_base,
        evaluation_strategy="steps",
    )


def mlflow_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple, set)):
        return ",".join(str(item) for item in value)
    if isinstance(value, torch.dtype):
        return str(value)
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def log_params(prefix, params):
    mlflow.log_params({f"{prefix}{key}": mlflow_value(value) for key, value in params.items()})


class MLflowTrainLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or mlflow.active_run() is None:
            return

        metrics = {}
        if "loss" in logs:
            metrics["train_loss"] = float(logs["loss"])
        if "learning_rate" in logs:
            metrics["learning_rate"] = float(logs["learning_rate"])
        if "grad_norm" in logs:
            metrics["grad_norm"] = float(logs["grad_norm"])

        if metrics:
            mlflow.log_metrics(metrics, step=state.global_step)

MAX_VALIDATION_SAMPLES = 1000

generator = torch.Generator().manual_seed(SEED)

all_indices = torch.randperm(
    len(raw_train_dataset),
    generator=generator,
).tolist()

validation_size = min(
    MAX_VALIDATION_SAMPLES,
    max(1, int(len(raw_train_dataset) * 0.05)),
)

validation_indices = all_indices[:validation_size]
train_indices = all_indices[validation_size:]

if MAX_TRAIN_SAMPLES is not None:
    train_indices = train_indices[:MAX_TRAIN_SAMPLES]

train_dataset = Subset(
    raw_train_dataset,
    train_indices,
)

validation_dataset = Subset(
    raw_train_dataset,
    validation_indices,
)

data_collator = QwenVQACollator(processor=processor, max_length=MAX_LENGTH)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    callbacks=[MLflowTrainLossCallback()],
)

if not globals().get("MLFLOW_READY", False):
    raise RuntimeError("Сначала выполните раздел подключения DagsHub и MLflow.")

STARTED_MLFLOW_RUN = False
if mlflow.active_run() is None:
    mlflow.start_run(run_name=RUN_NAME)
    STARTED_MLFLOW_RUN = True

log_params("", {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "model_source": MODEL_SOURCE,
    "data_source": DATA_SOURCE_INFO["data_source"],
    "num_train_examples": len(train_dataset),
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "checkpoint_dir": CHECKPOINT_DIR,
    "adapter_output_dir": OUTPUT_DIR,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "bf16": BF16,
    "fp16": FP16,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "optimizer": OPTIMIZER,
})
log_params("lora_", LORA_CONFIG)

prep_df = pd.DataFrame([
    {"Параметр": "run_name", "Значение": RUN_NAME},
    {"Параметр": "effective_batch_size", "Значение": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS},
    {"Параметр": "bf16", "Значение": BF16},
    {"Параметр": "fp16", "Значение": FP16},
])
display(prep_df)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


,Параметр,Значение
0,run_name,qwen2_5_vl_lora_seed_42
1,effective_batch_size,8
2,bf16,True
3,fp16,False


## 9. Запуск обучения

In [9]:
checkpoints = list(CHECKPOINT_DIR.glob("checkpoint-*"))

if checkpoints:
    latest_checkpoint = max(
        checkpoints,
        key=lambda path: int(path.name.split("-")[-1]),
    )
    resume_from_checkpoint = str(latest_checkpoint)
    print(f"Продолжаем обучение с чекпоинта: {latest_checkpoint}")
else:
    resume_from_checkpoint = None
    print("Чекпоинты не найдены. Начинаем обучение с нуля.")

train_result = trainer.train(
    resume_from_checkpoint=resume_from_checkpoint,
)

trainer.save_state()

train_metrics = {
    key: value
    for key, value in train_result.metrics.items()
    if isinstance(value, (int, float)) and math.isfinite(value)
}

trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)

if mlflow.active_run() is not None:
    mlflow.log_metrics(
        {
            f"train_{key}": float(value)
            for key, value in train_metrics.items()
        },
        step=trainer.state.global_step,
    )

pd.DataFrame([
    {"Метрика": key, "Значение": value}
    for key, value in train_metrics.items()
])

Чекпоинты не найдены. Начинаем обучение с нуля.


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
250,0.423493,0.435477
500,0.397126,0.387449
750,0.407153,0.370573
1000,0.355704,0.364262
1250,0.450346,0.348298
1500,0.397041,0.340766
1750,0.311451,0.332945
2000,0.297411,0.327664
2250,0.363258,0.323261
2500,0.298396,0.323591


***** train metrics *****
  epoch                    =         1.0
  total_flos               = 216546565GF
  train_loss               =      0.3462
  train_runtime            =  8:49:32.88
  train_samples_per_second =       0.944
  train_steps_per_second   =       0.118


,Метрика,Значение
0,train_runtime,3.177288e+04
1,train_samples_per_second,9.440000e-01
2,train_steps_per_second,1.180000e-01
3,total_flos,2.325151e+17
4,train_loss,3.462104e-01
5,epoch,1.000000e+00


После 3 тысяч улучшение минимально

## 10. Сохранение LoRA-адаптера

In [10]:
ADAPTER_OUTPUT_DIR = OUTPUT_DIR
ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_OUTPUT_DIR)
processor.save_pretrained(ADAPTER_OUTPUT_DIR)

serializable_lora_config = {
    key: mlflow_value(value)
    for key, value in LORA_CONFIG.items()
}
training_config = {
    "model_name": MODEL_NAME,
    "model_source": mlflow_value(MODEL_SOURCE),
    "data_source": DATA_SOURCE_INFO,
    "prompt_template": PROMPT_TEMPLATE,
    "seed": SEED,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_length": MAX_LENGTH,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "training": {
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "bf16": BF16,
        "fp16": FP16,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "save_steps": SAVE_STEPS,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "optimizer": OPTIMIZER,
    },
    "lora": serializable_lora_config,
    "train_metrics": globals().get("train_metrics", {}),
}

TRAINING_CONFIG_PATH = ADAPTER_OUTPUT_DIR / "training_config.json"
with TRAINING_CONFIG_PATH.open("w", encoding="utf-8") as file:
    json.dump(training_config, file, ensure_ascii=False, indent=2)

saved_files_df = pd.DataFrame([
    {"Артефакт": "LoRA adapter", "Путь": str(ADAPTER_OUTPUT_DIR)},
    {"Артефакт": "processor/tokenizer", "Путь": str(ADAPTER_OUTPUT_DIR)},
    {"Артефакт": "training_config.json", "Путь": str(TRAINING_CONFIG_PATH)},
])
display(saved_files_df)


,Артефакт,Путь
0,LoRA adapter,c:\Users\user\Desktop\Практика\vk-vision-langu...
1,processor/tokenizer,c:\Users\user\Desktop\Практика\vk-vision-langu...
2,training_config.json,c:\Users\user\Desktop\Практика\vk-vision-langu...


## 11. Логирование артефактов в DagsHub

In [11]:
if mlflow.active_run() is None:
    raise RuntimeError("Нет активного MLflow run. Выполните разделы подготовки и обучения перед логированием артефактов.")

mlflow.log_param("adapter_output_dir", str(ADAPTER_OUTPUT_DIR))
mlflow.log_artifacts(str(ADAPTER_OUTPUT_DIR), artifact_path="qwen_lora_adapter_3")
mlflow.log_artifact(str(TRAINING_CONFIG_PATH), artifact_path="training_config")

run_info = mlflow.active_run().info
artifact_df = pd.DataFrame([
    {"Параметр": "run_id", "Значение": run_info.run_id},
    {"Параметр": "artifact_uri", "Значение": run_info.artifact_uri},
    {"Параметр": "adapter_artifact_path", "Значение": "qwen_lora_adapter_3"},
])
display(artifact_df)

if globals().get("STARTED_MLFLOW_RUN", False):
    mlflow.end_run()


,Параметр,Значение
0,run_id,3ad86532bffa410a90a629afcecc5344
1,artifact_uri,mlflow-artifacts:/70cdf3873fdb434e81257920317e...
2,adapter_artifact_path,qwen_lora_adapter_3


🏃 View run qwen2_5_vl_lora_seed_42 at: https://dagshub.com/Dezurn/vk-vision-language-vqa.mlflow/#/experiments/5/runs/3ad86532bffa410a90a629afcecc5344
🧪 View experiment at: https://dagshub.com/Dezurn/vk-vision-language-vqa.mlflow/#/experiments/5


## 12. Выводы

Qwen LoRA #3 обучалась на 30 000 примерах одну эпоху и завершилась с `train_loss = 0,3462` за 8 ч 49 мин. LoRA охватила 161 визуальный модуль; обучаемая часть составила 82,25 млн параметров, или 2,14% модели. Адаптер и конфигурация сохранены.

В итоговой полной оценке на 12 216 примерах GQA-ru адаптер получил 0,5352 против 0,3198 у baseline. Результат выше LoRA #1 (0,5006) и на 0,20 п.п. ниже LoRA #2 (0,5372). На полном MMBench-ru LoRA #3 стала лучшей моделью с 0,8061 Accuracy.
